<a href="https://colab.research.google.com/github/meenakshim7/STUDENT-DB/blob/main/day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

print("LangChain imports successful")

LangChain imports successful


In [3]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("API key loaded successfully")

API key loaded successfully


In [4]:
import sqlite3

conn = sqlite3.connect("students.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Amirth", "AIDS", 92, 88, 95, 90),
    ("22CS048", "Arun", "AIDS", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88)
]

cursor.executemany("""
INSERT OR REPLACE INTO students
(student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", students)

conn.commit()

print("Database created successfully!")

Database created successfully!


In [5]:
cursor.execute("SELECT * FROM students")

rows = cursor.fetchall()

for row in rows:
    print(row)

('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78)
('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72)
('22CS047', 'Amirth', 'AIDS', 92, 88, 95, 90)
('22CS048', 'Arun', 'AIDS', 55, 60, 58, 62)
('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)


In [6]:
from langchain_core.tools import tool

In [7]:
@tool
def get_student_info(student_id: str) -> dict:
    """Get the name and department of a student using their student ID."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT name, department
        FROM students
        WHERE student_id = ?
    """, (student_id,))

    result = cursor.fetchone()

    conn.close()

    if result is None:
        return {
            "error": "Student not found"
        }

    return {
        "name": result[0],
        "department": result[1]
    }

In [8]:
result = get_student_info.invoke({
    "student_id": "22CS045"
})

print(result)

{'name': 'Dhanushya', 'department': 'Computer Science'}


In [9]:
@tool
def get_student_marks(student_id: str) -> dict:
    """Get the Python, Database, AI, and Web marks of a student."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT python, database, ai, web
        FROM students
        WHERE student_id = ?
    """, (student_id,))

    result = cursor.fetchone()

    conn.close()

    if result is None:
        return {
            "error": "Student not found"
        }

    return {
        "python": result[0],
        "database": result[1],
        "ai": result[2],
        "web": result[3]
    }

In [10]:
result = get_student_marks.invoke({
    "student_id": "22CS045"
})

print(result)

{'python': 85, 'database': 72, 'ai': 90, 'web': 78}


In [11]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression such as 85+72+90+78 or (85+72+90+78)/4."""

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)

    except Exception:
        return "Invalid mathematical expression"

In [12]:
@tool
def get_passing_rules() -> dict:
    """Get the university passing requirements."""

    return {
        "minimum_overall_average": 40,
        "minimum_subject_mark": 35
    }

In [13]:
print(
    get_student_info.invoke({
        "student_id": "22CS045"
    })
)

{'name': 'Dhanushya', 'department': 'Computer Science'}


In [14]:
print(
    calculator.invoke({
        "expression": "85+72+90+78"
    })
)


325


In [15]:
print(get_passing_rules.invoke({}))

{'minimum_overall_average': 40, 'minimum_subject_mark': 35}
